In [ ]:
import pandas as pd
from pathlib import Path
from evaluations.aggregate_metrics import aggregate
from evaluations import labeled_data_matches, space_groups
from evaluations.eval import relax
from phonons.phonons import stability_screening
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import read, write

import sys

sys.path.append("./evaluations")

In [ ]:
df = aggregate(
    Path("generated_structures/topological_dgf2"),
    Path(
        "predictive_models/topological_mp_20/inference/predictions/topological_dgf2/predictions.csv"
    ),
)[:20_000]

In [ ]:
df.columns

In [ ]:
allowed_space_groups = {
    2,
    5,
    6,
    7,
    10,
    11,
    12,
    13,
    14,
    15,
    19,
    26,
    29,
    36,
    43,
    46,
    47,
    51,
    52,
    53,
    55,
    57,
    58,
    59,
    60,
    62,
    63,
    64,
    65,
    68,
    69,
    70,
    71,
    72,
    74,
    82,
    84,
    87,
    97,
    107,
    113,
    114,
    115,
    119,
    120,
    121,
    122,
    123,
    125,
    127,
    128,
    129,
    131,
    136,
    137,
    139,
    140,
    141,
    142,
    146,
    148,
    155,
    156,
    161,
    162,
    163,
    164,
    165,
    166,
    167,
    176,
    180,
    182,
    185,
    186,
    187,
    189,
    190,
    191,
    193,
    194,
    198,
    199,
    205,
    206,
    212,
    213,
    215,
    216,
    217,
    220,
    221,
    223,
    225,
    226,
    227,
    229,
}
print(f"Restrict to {len(allowed_space_groups)} space groups from TopoHSE dataset.")


def get_criteria(df):
    criteria = (
        (df["unique"])
        & (df["predicted_label"] == 1)
        & (df["in_mp"] == False)
        & (df["structure_comp_validity"])
        & (df["energy_above_hull"] < 0.10)
        & (df["num_elements"].between(2, 4, inclusive="both"))
        & (df["p-block"])
        & (df["space_group_number_001"].isin(allowed_space_groups))
    )

    return criteria

In [ ]:
candidates = df[get_criteria(df)].copy()
candidates[["formula"]]

In [ ]:
output_dir = Path("generated_structures/topological_dgf2")

In [ ]:
# save candidates to extxyz file
candidates_file = output_dir / "candidates.extxyz"

if candidates_file.exists():
    print(f"{candidates_file} already exists, skipping write.")
else:
    candidate_structures = [AseAtomsAdaptor.get_atoms(s) for s in candidates["structure"]]
    write(candidates_file, candidate_structures)

In [ ]:
# Set potential for relaxation and forces prediction for phonon calculations
potential_load_path = "MatterSim-v1.0.0-5M.pth"

In [ ]:
# relax candidates to a higher accuracy using MatterSim, (e.g. forces)
# potentially run this from a separate script, as it can take a while
if (output_dir / "candidates_highly_relaxed.extxyz").exists():
    print(f"Relaxed structures already exist in {output_dir}, skipping relaxation.")
else:
    relax(output_dir, fmax=1e-5, potential_load_path=potential_load_path)

In [ ]:
# Load relaxed structures
ase_atoms_list = read(output_dir / "candidates_highly_relaxed.extxyz", index=":")
relaxed_candidate_structures = [AseAtomsAdaptor.get_structure(atoms) for atoms in ase_atoms_list]
candidates["relaxed_candidate_structure"] = relaxed_candidate_structures

In [ ]:
# Now run phonon calculations on the relaxed structures
# This may also take a while

output_filename = output_dir / "phonon_stability_results.csv"

if output_filename.exists():
    print(f"{output_filename} already exists, skipping phonon stability screening.")
else:
    _ = stability_screening(
        structures=relaxed_candidate_structures,
        potential_load_path=potential_load_path,
        output_file=output_filename,
    )

In [ ]:
# Read the results from csv
phonon_df = pd.read_csv(output_filename)
phonon_df.head()

In [ ]:
stable_candidates = candidates.copy().reset_index(drop=True)

# Filter for dynamically stable candidates using phonon frequency threshold
stable_mask = (
    phonon_df["min_frequency_THz"] > -0.1
)  # use -0.1 THz as threshold to account for numerical noise
stable_candidates = stable_candidates[stable_mask]
stable_candidates[["formula"]]

In [ ]:
columns = [
    "formula",
    "structure",
    "energy_above_hull_per_atom",
    "self_consistent_energy_above_hull",
    "space_group_01",
    "space_group_001",
    "space_group_number_01",
    "space_group_number_001",
    "matches_in_reference",
    "rmsd_from_relaxation",
    "novel_unique_stable",
    "structure_comp_validity",
]

dft_candidates = stable_candidates.copy()
dft_candidates["structure"] = dft_candidates["relaxed_candidate_structure"]
dft_candidates["space_group_info_01"] = dft_candidates["structure"].map(
    lambda s: s.get_space_group_info(symprec=0.1)
)
dft_candidates["space_group_info_001"] = dft_candidates["structure"].map(
    lambda s: s.get_space_group_info(symprec=0.01)
)
dft_candidates["space_group_01"] = dft_candidates["space_group_info_01"].map(lambda x: x[0])
dft_candidates["space_group_001"] = dft_candidates["space_group_info_001"].map(lambda x: x[0])
dft_candidates["space_group_number_01"] = dft_candidates["space_group_info_01"].map(lambda x: x[1])
dft_candidates["space_group_number_001"] = dft_candidates["space_group_info_001"].map(
    lambda x: x[1]
)

# turn structures into cif strings
dft_candidates["structure"] = dft_candidates["structure"].map(lambda s: s.to(fmt="cif"))

In [ ]:
# dft_candidates[columns].to_csv("dft_candidates.csv", index=False)
dft_candidates